# Step 9: The real ML service
## Part A: The history the service keeps
The model compares each request with the employee's, department's and category's earlier requests.
The ML service can't read the database, so it keeps its own copy of the history (`src/history.py`):
the training requests, plus every request it scores. This part builds it, saves it, and checks that it
gives exactly the same features as `build_features` (training and serving must match, MF1).

In [1]:
# 9.1 Load the requests, the shared feature code, the new history code, and the model settings.
import json
import os
import sys
import time
import numpy as np
import pandas as pd

sys.path.insert(0, "..")                 # lets the notebook import from ml/src
from src import features as F
from src.history import History          # NEW (Step 9): the history the service keeps

requests = pd.read_csv("../data/processed/requests.csv", parse_dates=["expense_date"])
with open("../models/model_config.json", encoding="utf-8") as fh:
    cfg = json.load(fh)
print(cfg["model_version"], "|", len(requests), "requests |", requests["expense_date"].min().date(), "→", requests["expense_date"].max().date())

iforest-0.2 | 247838 requests | 2025-05-14 → 2026-06-28


_Code for this step comes next._

### 9.2 Move the history dates to the present
The history ends in June 2026, but the demo is in Nov/Dec 2026. Without a move, every employee's first
demo request would be "the first request after 90+ days". Everything moves by exactly 21 weeks, so every
weekday stays the same and every gap between requests stays the same → the features don't change.
The seed files for the backend (later in Step 9) use the same move.

In [2]:
# 9.2 Move every date by whole weeks (weekdays and gaps unchanged), so the history ends just before the demo.
SHIFT_WEEKS = 21
shifted = requests.assign(expense_date=requests["expense_date"] + pd.Timedelta(weeks=SHIFT_WEEKS))
print("History after the shift:", shifted["expense_date"].min().date(), "→", shifted["expense_date"].max().date())
print("Same weekday for every request:", (shifted["expense_date"].dt.day_of_week == requests["expense_date"].dt.day_of_week).all())

# proof that the model sees the same thing: every feature (except the date itself) is identical
before = F.build_features(requests)
after = F.build_features(shifted)
same_cols = [c for c in before.columns if c != "expense_date"]
print("Features identical after the shift:", before[same_cols].equals(after[same_cols]))

History after the shift: 2025-10-08 → 2026-11-22
Same weekday for every request: True
Features identical after the shift: True


In [3]:
# 9.3 Build the history (bulk buyers left out, like training), save it next to the model, reload it.
history = History.from_requests(shifted)
history.save("../models/history.joblib")
reloaded = History.load("../models/history.joblib")
print(len(history.rows), "requests kept |", requests["requester_id"].nunique() - history.rows["requester_id"].nunique(), "bulk buyers left out")
print("File size:", round(os.path.getsize("../models/history.joblib") / 1e6, 2), "MB | reloaded identical:", reloaded.rows.equals(history.rows))

209012 requests kept | 13 bulk buyers left out
File size: 1.5 MB | reloaded identical: True


### 9.4 Training vs serving check (MF1)
For ~1,600 requests (random + the tricky ones: splits, repeats within 7 days, new employees), compute the
features one request at a time, the way the service will, and compare with `build_features`.
Every value must be the same.

In [4]:
# 9.4 One request at a time (like the service) vs the whole table at once (like training).
# The service gets the same-day requests from the backend; here we take them from the history.
sample = pd.concat([
    after.sample(1000, random_state=9),                                                    # random
    after[after["same_category_day_count"] >= 3].sample(200, random_state=1),              # busy days / splits
    after[after["days_since_same_amount"].between(0, 7)].sample(200, random_state=2),      # same amount again soon
    after[after["baseline_level"].isin(["deptcat", "cat", "none"])].sample(200, random_state=3),  # new employees
]).drop_duplicates("expense_id")
typed = shifted.set_index("expense_id")["amount_usd"]      # the amount as written (USD in history) for round/digits
days = history.rows.groupby(["requester_id", "expense_date"])

start = time.time()
served = []
for _, r in sample.iterrows():
    day = days.get_group((r["requester_id"], r["expense_date"]))
    others = day[day["expense_id"] != r["expense_id"]][["expense_id", "category", "amount_sar"]]   # = same_day_requests
    served.append(history.request_features(
        r["expense_id"], r["requester_id"], r["department_id"], r["category"], r["amount_sar"],
        r["expense_date"], others.to_dict("records"), typed_amount=typed[r["expense_id"]]))
served = pd.concat(served).set_index("expense_id")
trained = sample.set_index("expense_id").loc[served.index]
print(len(served), "requests in", round(time.time() - start, 1), "s")

check = F.FEATURES + F.RULE_COLUMNS
wrong = {c: int((~np.isclose(served[c].astype(float), trained[c].astype(float), atol=1e-9, equal_nan=True)).sum()) for c in check}
wrong["baseline_level"] = int((served["baseline_level"] != trained["baseline_level"]).sum())
print("Mismatches per column:", {c: n for c, n in wrong.items() if n} or "none")
print("Total mismatches:", sum(wrong.values()), "of", len(served) * len(wrong), "values")

1599 requests in 27.3 s
Mismatches per column: none
Total mismatches: 0 of 23985 values


In [5]:
# 9.5 Two checks of how the service will behave:
#  (a) a brand-new employee (id 99999, department 1, no history) → compared with the department (cold start)
#  (b) why the date move matters: share of employees whose first demo request would be a "90+ day break"
example = history.request_features(990001, 99999, 1, "Food", 450.00, "2026-11-25")
print(example[["baseline_level", "amount_vs_employee_category", "category_share_for_employee",
               "days_since_last_request"]].round(3).to_string(index=False))

last_request = History.from_requests(requests).rows.groupby("requester_id")["expense_date"].max()
last_shifted = history.rows.groupby("requester_id")["expense_date"].max()
demo_day = pd.Timestamp("2026-11-25")
print("90+ days since their last request on", demo_day.date(), "→ without the shift:",
      round(((demo_day - last_request).dt.days >= 90).mean() * 100, 1), "% | with it:",
      round(((demo_day - last_shifted).dt.days >= 90).mean() * 100, 1), "%")
real_end = pd.Timestamp("2026-06-28")
print("On the last day of the real history:", round(((real_end - last_request).dt.days >= 90).mean() * 100, 1), "%")

baseline_level  amount_vs_employee_category  category_share_for_employee  days_since_last_request
       deptcat                        0.687                        0.193                      NaN
90+ days since their last request on 2026-11-25 → without the shift: 100.0 % | with it: 21.1 %
On the last day of the real history: 20.5 %


In [6]:
# 9.6 Write the history settings into model_config.json, so the service knows which file to load and how
# the dates were moved (the seed files must use the same move).
cfg["history"] = {
    "file": "history.joblib",
    "date_shift_weeks": SHIFT_WEEKS,
    "from": str(history.rows["expense_date"].min().date()),
    "to": str(history.rows["expense_date"].max().date()),
    "requests": int(len(history.rows)),
}
with open("../models/model_config.json", "w", encoding="utf-8") as fh:
    json.dump(cfg, fh, indent=2)
print(json.dumps(cfg["history"], indent=2))

{
  "file": "history.joblib",
  "date_shift_weeks": 21,
  "from": "2025-10-08",
  "to": "2026-11-22",
  "requests": 209012
}


## Part B: The real scoring (`src/predict.py`)
The stub is replaced by the real model, with the same inputs and outputs. For one request: features from the
history → model score → the 6 rules (with the category limit the backend sends) → risk level → reasons →
the request is added to the history. The reason settings from Step 8 move into `model_config.json`, so the
service uses exactly what was evaluated.

In [7]:
# 9.7 Save the reason settings from Step 8 (cells 8.3–8.4) in model_config.json, so the service uses the same ones.
# normal_values = what a typical request has (the "what if" test sets one feature back to this).
train = pd.read_csv("../data/processed/features.csv", parse_dates=["expense_date"])
X_train = F.model_input(train, cfg)
cfg["reasons"] = {
    "normal_values": {
        "log_amount": cfg["model_input"]["log_amount_floor"],
        "amount_vs_employee_category": 0.0,
        "category_share_for_employee": float(X_train["category_share_for_employee"].median()),
        "is_weekend": 0,
        "is_round_amount": 0,
    },
    "min_drop": 0.02,        # a feature is a reason if setting it to normal lowers the score by at least this
    "max_reasons": 3,        # the contract's limit
    "unusual_share": 0.2,    # "rare category" = under 20% of this person's earlier requests
    "min_spreads": 1,        # the amount reason needs the amount at least 1 spread above this person's normal
}
with open("../models/model_config.json", "w", encoding="utf-8") as fh:
    json.dump(cfg, fh, indent=2)
print(json.dumps(cfg["reasons"], indent=2))

{
  "normal_values": {
    "log_amount": 5.8661,
    "amount_vs_employee_category": 0.0,
    "category_share_for_employee": 0.5886406224056548,
    "is_weekend": 0,
    "is_round_amount": 0
  },
  "min_drop": 0.02,
  "max_reasons": 3,
  "unusual_share": 0.2,
  "min_spreads": 1
}


In [8]:
# 9.8 Load the real scoring code. It reads the model, the config and the history when it's imported,
# so reload it whenever model_config.json changes.
import importlib
import src.predict as P
importlib.reload(P)
print(P.MODEL_VERSION, "|", len(P.REASONS), "reason codes |", len(P.HISTORY.rows), "requests in the history |", P.CATEGORIES == list(cfg["category_limits"]))

iforest-0.2 | 13 reason codes | 209012 requests in the history | True


### 9.9 Service vs batch check
~900 requests (400 flagged, 400 random, 100 splits) scored one at a time with `score_request`, the way the
service will, compared with the batch scoring from Step 8 (whole table at once). Score, risk level and reasons
must all be the same. `remember=False` so the check doesn't add anything to the history.

In [9]:
# 9.9 One request at a time (the service) vs the whole table at once (Step 8).
scores = -P.MODEL.score_samples(F.model_input(after, cfg))       # batch: every request at once
rules_all = F.apply_rules(after, cfg)
levels = F.risk_levels(scores, rules_all, cfg)
flagged_idx = np.flatnonzero(levels != "low")
pick = np.unique(np.concatenate([
    np.random.default_rng(1).choice(flagged_idx, 400, replace=False),                  # flagged
    np.random.default_rng(2).choice(len(after), 400, replace=False),                   # random
    np.flatnonzero(rules_all["possible_split_request"].to_numpy())[:100],             # splits
]))
start = time.time()
diff = {"score": 0, "risk_level": 0, "reasons": 0}
for i in pick:
    r = after.iloc[i]
    day = days.get_group((r["requester_id"], r["expense_date"]))
    others = [{"expense_id": o.expense_id, "category": o.category, "amount": o.amount_sar}     # = same_day_requests
              for o in day.itertuples() if o.expense_id != r["expense_id"]]
    expense = {"expense_id": int(r["expense_id"]), "requester_id": int(r["requester_id"]),
               "department_id": int(r["department_id"]), "category": r["category"],
               "amount": float(r["amount_sar"]), "expense_date": str(r["expense_date"].date())}
    got = P.score_request(expense, others, remember=False, typed_amount=typed[r["expense_id"]])
    want_codes = []
    if levels[i] != "low":                                         # batch reasons, the 8.4 way
        X1 = F.model_input(after.iloc[[i]], cfg)
        want_codes = P.explain(r, P.score_and_drops(X1.reset_index(drop=True))[1], rules_all.iloc[i])
    diff["score"] += not np.isclose(got["score"], scores[i], rtol=0, atol=1e-12)
    diff["risk_level"] += got["risk_level"] != levels[i]
    diff["reasons"] += [d["code"] for d in got["reason_details"]] != want_codes
print(len(pick), "requests,", (levels[pick] != "low").sum(), "flagged |", round((time.time() - start) / len(pick) * 1000, 1), "ms per request")
print("Different from the batch scoring:", diff)

896 requests, 512 flagged | 58.3 ms per request
Different from the batch scoring: {'score': 0, 'risk_level': 0, 'reasons': 0}


### 9.10 What the backend will get: 4 example requests
Requests dated after the history (Nov 2026), scored exactly as the service will score them.
Only the last pair uses `remember=True` (the second request must see the first); nothing is saved to disk.

In [10]:
# 9.10 Example requests through score_request, the way the service will get them.
def ask(expense_id, employee, department, category, amount, date, same_day=(), limit=None, remember=False):
    expense = {"expense_id": expense_id, "requester_id": employee, "department_id": department,
               "category": category, "amount": amount, "expense_date": date, "category_limit": limit}
    out = P.score_request(expense, list(same_day), remember=remember)
    print(f"{expense_id} {category:9s} {amount:>9,.2f} {date} → {out['risk_level']:6s} {out['score']:.3f}",
          [d["code"] for d in out["reason_details"]])
    return out

print("Employee 1734 (department 1, usually Food around 58 SAR):")
ask(990001, 1734, 1, "Food", 55.00, "2026-11-25")
ask(990002, 1734, 1, "Food", 1850.00, "2026-11-27")            # a Friday

print("\nEmployee 3639 (department 20, Food around 230 SAR): 3 Food requests on one day, total 2,120 SAR")
day_parts = [{"expense_id": 990011, "category": "Food", "amount": 750.00},
             {"expense_id": 990012, "category": "Food", "amount": 680.00}]
ask(990013, 3639, 20, "Food", 690.00, "2026-11-25", day_parts)               # ML's Food limit: 2,000
ask(990013, 3639, 20, "Food", 690.00, "2026-11-25", day_parts, limit=3000)   # backend says the limit is 3,000 now

print("\nEmployee 3639: the same Travel amount twice, 3 days apart")
ask(990021, 3639, 20, "Travel", 2340.50, "2026-11-23", remember=True)
ask(990022, 3639, 20, "Travel", 2340.50, "2026-11-26")

Employee 1734 (department 1, usually Food around 58 SAR):
990001 Food          55.00 2026-11-25 → low    0.383 []
990002 Food       1,850.00 2026-11-27 → high   0.690 ['weekend_expense', 'amount_high_for_employee_category']

Employee 3639 (department 20, Food around 230 SAR): 3 Food requests on one day, total 2,120 SAR
990013 Food         690.00 2026-11-25 → medium 0.469 ['possible_split_request', 'amount_high_for_employee_category']
990013 Food         690.00 2026-11-25 → low    0.469 []

Employee 3639: the same Travel amount twice, 3 days apart
990021 Travel     2,340.50 2026-11-23 → low    0.497 []
990022 Travel     2,340.50 2026-11-26 → medium 0.492 ['duplicate_recent_request', 'unusual_category_for_employee']


{'expense_id': 990022,
 'score': 0.4924071344657459,
 'risk_level': 'medium',
 'is_flagged': True,
 'reasons': ['Same amount and category as a request a few days earlier (possible receipt used twice)',
  'This employee rarely submits requests in this category'],
 'reason_details': [{'code': 'duplicate_recent_request',
   'text_en': 'Same amount and category as a request a few days earlier (possible receipt used twice)',
   'text_ar': 'نفس المبلغ والفئة لطلب قبل أيام قليلة (احتمال استخدام الإيصال مرتين)'},
  {'code': 'unusual_category_for_employee',
   'text_en': 'This employee rarely submits requests in this category',
   'text_ar': 'نادراً ما يقدّم هذا الموظف طلبات في هذه الفئة'}],
 'model_version': 'iforest-0.2'}

## Part C: The service
`src/service.py` now runs the real model. New: the optional `category_limit` (contract v0.4.5), `/health` shows
the history, and every scored request is saved to `data/service/added_requests.csv` (git-ignored) so a restart
doesn't forget it. Tested here with FastAPI's TestClient, which calls the service without starting a server.

In [2]:
# 9.11 Re-save the history in the new format (plain numpy columns, so it loads with any pandas version).
import joblib
from src.history import COLUMNS
path = "../models/history.joblib"
old = History.load(path)
old.save(path)
new = History.load(path)
print("Saved as:", type(joblib.load(path)).__name__, "| same requests:", new.rows[COLUMNS].equals(old.rows[COLUMNS]),
      "|", round(os.path.getsize(path) / 1e6, 2), "MB")

Saved as: dict | same requests: True | 1.5 MB


In [3]:
# 9.12 Call the service the way the backend will (HTTP + JSON), without starting a server.
from fastapi.testclient import TestClient
import src.service as S
client = TestClient(S.app)
print("GET /health →", client.get("/health").json())

def post(body):
    r = client.post("/anomaly/score", json=body)
    if r.status_code != 200:
        return r.status_code, r.json()["detail"][0]["msg"]          # the error the backend would get
    out = r.json()
    return r.status_code, out["risk_level"], [d["code"] for d in out["reason_details"]]

normal = {"expense_id": 990001, "requester_id": 1734, "department_id": 1, "category": "Food",
          "amount": "55.00", "expense_date": "2026-11-25", "same_day_requests": []}
print("normal request      →", post(normal))
print("unknown category    →", post({**normal, "expense_id": 990002, "category": "Other"}))
parts = [{"expense_id": 990011, "category": "Food", "amount": "750.00"},
         {"expense_id": 990012, "category": "Food", "amount": "680.00"}]
split = {"expense_id": 990013, "requester_id": 3639, "department_id": 20, "category": "Food",
         "amount": "690.00", "expense_date": "2026-11-25", "same_day_requests": parts}
print("split, no limit     →", post(split))
print("split, limit 3000   →", post({**split, "category_limit": "3000.00"}))

# the biggest call the backend can make: 500 requests at once (its timeout is 30 s, contract C2)
items = [{**normal, "expense_id": 991000 + i, "amount": f"{40 + i % 50}.25", "expense_date": "2026-11-28"} for i in range(500)]
start = time.time()
r = client.post("/anomaly/score-batch", json={"items": items})
print("batch of 500        →", r.status_code, len(r.json()["results"]), "results in", round(time.time() - start, 1), "s")
print("GET /health →", client.get("/health").json()["history"])

GET /health → {'status': 'ok', 'model_version': 'iforest-0.2', 'history': {'requests': 209012, 'added_since_training': 0, 'latest_date': '2026-11-22'}}
normal request      → (200, 'low', [])
unknown category    → (422, "Input should be 'Food', 'Travel', 'Transportation', 'Office Supplies', 'IT & Telecom' or 'Equipment & Maintenance'")
split, no limit     → (200, 'medium', ['possible_split_request', 'amount_high_for_employee_category'])
split, limit 3000   → (200, 'low', [])
batch of 500        → 200 500 results in 26.9 s
GET /health → {'requests': 209514, 'added_since_training': 502, 'latest_date': '2026-11-28'}


In [6]:
# 9.13 Remove the test requests, so the real service starts from the training history only.
added = "../data/service/added_requests.csv"
print("Test requests in the file:", len(pd.read_csv(added)))
os.remove(added)
print("Removed:", not os.path.exists(added))

FileNotFoundError: [Errno 2] No such file or directory: '../data/service/added_requests.csv'